# 09v1 - A2 v2: multi-group slot attention (test hypothesis 2)

**Plan item:** follow-up to A2 v1, testing whether using all 3 A3 anchor
groups (18 pseudo-slots) instead of centre-only (`group_index=1`, 6
slots) improves the weak-finding cluster
(`mcl_injury`/`medial_meniscus_tear`/`lateral_meniscus_tear`/
`oa_lateral_compartment`). Full design, evidence, and the gate this run
is measured against:
docs/superpowers/specs/2026-08-28-a2v2-multigroup-slot-attention-design.md
(approved after two Opus review passes).

Self-contained per this project's Kaggle constraint (no `import src`) -
`select_group()`/`expand_slot_groups()`/`SlotCacheDataset` below are
hand-kept notebook-local versions; per the approved spec section 6, none
of this graduates to `src/` until the user reviews this notebook's real
output.

**Scope: fold 0 only**, `group_index`/pseudo-slot design is the only
changed variable vs. A2 v1
(`notebooks/05v2_slot_attention_baseline.ipynb`) - same
hyperparameters, same labels, same fold split.

In [ ]:
import hashlib
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the full DICOM tree + GPU + the A3 cache "
        "attached - run on Kaggle, not locally."
    )
RAW_DIR = _KAGGLE_RAW
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")

# llm_labels_v4_blend.csv is NOT part of the official competition mount -
# it's the user's own separately-attached Kaggle Dataset. EDIT this path
# to match wherever it actually lands under /kaggle/input/ once attached
# (same convention as CACHE_DIR - check `!ls /kaggle/input` if unsure).
PUBLISHED_LABELS_PATH = Path("/kaggle/input/llm-labels-v4-blend/llm_labels_v4_blend.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("CACHE_DIR:", CACHE_DIR, "exists:", CACHE_DIR.exists())
print("PUBLISHED_LABELS_PATH:", PUBLISHED_LABELS_PATH, "exists:", PUBLISHED_LABELS_PATH.exists())

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
SLOT_NAMES = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOT_CACHE_GROUP_SIZE = 3
SLOT_CACHE_N_GROUPS = 3
N_SLOTS = len(SLOT_NAMES) * SLOT_CACHE_N_GROUPS  # 18 pseudo-slots -- A2 v2's own change
CV_FOLDS = 4
TRAIN_SHARDS = [f"train.s{i:02d}of04" for i in range(4)]

## Labels: gold official values + A1a' published set for weak studies

In [ ]:
def load_published_labels(path):
    published = pd.read_csv(path)
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    published = published.set_index("StudyInstanceUID")[label_cols]
    published.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return published


def load_gold_labels(raw_dir):
    train = pd.read_csv(raw_dir / "train.csv")
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    gold_mask = train[label_cols].notna().all(axis=1)
    gold = train.loc[gold_mask, ["StudyInstanceUID"] + label_cols].set_index("StudyInstanceUID")
    gold.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return gold


train_csv = pd.read_csv(RAW_DIR / "train.csv")
reports = train_csv.set_index("StudyInstanceUID")[["Report"]]
gold = load_gold_labels(RAW_DIR)
published = load_published_labels(PUBLISHED_LABELS_PATH)

missing = set(train_csv["StudyInstanceUID"]) - set(published.index)
print("train.csv studies missing from published labels:", len(missing))
assert len(missing) == 0

is_gold = reports.index.isin(gold.index)
label_table = published.reindex(reports.index)[FINDINGS].copy()
label_table.loc[gold.index, FINDINGS] = gold[FINDINGS]
label_table["is_gold"] = is_gold
print(label_table.shape, "gold rows:", label_table["is_gold"].sum())

## Folds: report-template + scanner-fingerprint grouping (A0)

Identical logic to `05v2`/`06v2`'s fold-assignment cell - `GroupKFold`
has no shuffling/randomness, so recomputing from the same inputs
reproduces the exact same split. **Asserted, not just assumed**, per
the approved spec section 6 (first-review finding: fold reproducibility
was previously unstated) - if this assertion ever fails, stop and
investigate before trusting any comparison against `05v2`'s 0.7689.

In [ ]:
def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


SCANNER_FINGERPRINT_TAGS = (
    "Manufacturer", "ManufacturerModelName", "InstitutionName",
    "DeviceSerialNumber", "MagneticFieldStrength", "StationName",
)


def build_scanner_fingerprints(raw_dir, split="train"):
    series = pd.read_csv(raw_dir / f"{split}_series.csv")
    first_series = series.drop_duplicates("StudyInstanceUID", keep="first")
    fingerprints = {}
    for row in first_series.itertuples(index=False):
        series_dir = raw_dir / f"{split}_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            fingerprints[row.StudyInstanceUID] = None
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        fingerprints[row.StudyInstanceUID] = tuple(
            str(getattr(ds, tag, None)) for tag in SCANNER_FINGERPRINT_TAGS
        )
    result = pd.Series(fingerprints, name="scanner_fingerprint")
    result.index.name = "StudyInstanceUID"
    return result


def build_group_ids(*group_key_series):
    index = group_key_series[0].index
    parent = {i: i for i in index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for keys in group_key_series:
        valid = keys.dropna()
        for _, idx in valid.groupby(valid).groups.items():
            idx = list(idx)
            for other in idx[1:]:
                union(idx[0], other)

    return pd.Series({i: find(i) for i in index}, name="group_id")


t0 = time.time()
scanner_fp = build_scanner_fingerprints(RAW_DIR, split="train")
print(f"scanner fingerprints: {time.time() - t0:.1f}s for {len(scanner_fp)} studies")

group_keys = reports["Report"].apply(report_group_key)
group_ids = build_group_ids(group_keys, scanner_fp.reindex(reports.index))

gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_ids.to_numpy())):
    fold.iloc[val_idx] = fold_idx
label_table["fold"] = fold
print(label_table["fold"].value_counts().sort_index())

fold0_val = label_table[label_table["fold"] == 0]
print(f"\nfold 0 val: {len(fold0_val)} studies, {int(fold0_val['is_gold'].sum())} gold")
assert len(fold0_val) == 1307, (
    f"fold 0 val set size {len(fold0_val)} != 05v2's recorded 1307 - fold split diverged, "
    "stop and investigate before trusting any comparison against 0.7689"
)
assert int(fold0_val["is_gold"].sum()) == 17, (
    f"fold 0 gold count {int(fold0_val['is_gold'].sum())} != 05v2's recorded 17"
)
print("fold 0 matches 05v2's recorded split exactly - OK, safe to compare against 0.7689")

## Reshape: `select_group()` (A3, unchanged) + `expand_slot_groups()` (A2 v2, new)

`select_group()` is an unchanged hand-kept copy of the graduated
`src/features.py` version. `expand_slot_groups()` is this experiment's
core data-layer change - per the approved spec section 2.1, reshapes
`(6 slots, 9 slices)` into `(18 pseudo-slots, 3 channels)`, slot-major
order, each real slot's mask bit replicated x3. Self-validated below
against a synthetic array before trusting it on real cache data - same
practice as `04v2`'s `select_group` demo cell.

In [ ]:
def select_group(cache_slot_stack, group_index):
    if isinstance(group_index, int):
        group_index = [group_index]
    size = SLOT_CACHE_GROUP_SIZE
    groups = [cache_slot_stack[..., g * size:(g + 1) * size, :, :] for g in group_index]
    return np.concatenate(groups, axis=-3)


def expand_slot_groups(cache_slot_stack, slot_mask):
    '''cache_slot_stack: (n_slots, 9, H, W) - one study, all slots.
    slot_mask: (n_slots,). Returns (images, mask):
    images (n_slots * SLOT_CACHE_N_GROUPS, SLOT_CACHE_GROUP_SIZE, H, W),
    slot-major order (pseudo-slot index = s * SLOT_CACHE_N_GROUPS + g).
    mask (n_slots * SLOT_CACHE_N_GROUPS,), each real slot's bit repeated
    SLOT_CACHE_N_GROUPS times. Per approved spec section 2.1.'''
    n_slots = cache_slot_stack.shape[0]
    h, w = cache_slot_stack.shape[-2:]
    groups = [select_group(cache_slot_stack, g) for g in range(SLOT_CACHE_N_GROUPS)]
    stacked = np.stack(groups, axis=1)  # (n_slots, n_groups, 3, H, W)
    images = stacked.reshape(n_slots * SLOT_CACHE_N_GROUPS, SLOT_CACHE_GROUP_SIZE, h, w)
    mask = np.repeat(slot_mask, SLOT_CACHE_N_GROUPS)
    return images, mask


# Self-validate: each channel's pixels are set to that channel's global
# index, so the expected output is exact and checkable by hand.
_demo_stack = np.zeros((6, 9, 4, 4), dtype=np.uint8)
for _c in range(9):
    _demo_stack[:, _c] = _c
_demo_mask = np.array([1.0, 0.0, 1.0, 1.0, 0.0, 1.0], dtype=np.float32)
_images, _mask = expand_slot_groups(_demo_stack, _demo_mask)
assert _images.shape == (18, 3, 4, 4)
assert _mask.shape == (18,)
for _s in range(6):
    for _g in range(3):
        assert np.array_equal(_images[_s * 3 + _g], select_group(_demo_stack[_s], _g))
assert np.array_equal(_mask, np.repeat(_demo_mask, 3))
print("expand_slot_groups matches select_group per (slot, group) pair and replicates the mask - OK")

## Cache dataset

Opens all 4 train shards as memmaps (never materialises the ~12GB cache
in RAM). New `expand_groups` parameter (default `False`, matching A2
v1's exact behaviour): when `True`, `__getitem__` calls
`expand_slot_groups()` instead of `select_group(group_index)`, and a
non-default `group_index` passed alongside `expand_groups=True` raises
(spec section 2.2 - avoids a silently-ignored argument).

In [ ]:
class SlotCacheDataset(torch.utils.data.Dataset):
    def __init__(self, cache_dir, shards, labels_df, group_index=1, expand_groups=False, study_ids=None):
        if expand_groups and group_index != 1:
            raise ValueError(
                "expand_groups=True ignores group_index; pass group_index=1 (default) or omit "
                f"it, not group_index={group_index!r}"
            )
        self.group_index = group_index
        self.expand_groups = expand_groups
        caches, masks, all_study_ids, shard_of, local_idx = [], [], [], [], []
        for shard in shards:
            cache = np.load(cache_dir / f"{shard}_cache.npy", mmap_mode="r")
            mask = np.load(cache_dir / f"{shard}_mask.npy")
            studies = pd.read_csv(cache_dir / f"{shard}_studies.csv")
            caches.append(cache)
            masks.append(mask)
            all_study_ids.append(studies["StudyInstanceUID"].to_numpy())
            shard_of.append(np.full(len(studies), len(caches) - 1))
            local_idx.append(np.arange(len(studies)))

        self.caches = caches
        mask_all = np.concatenate(masks, axis=0).astype(np.float32)
        study_ids_all = np.concatenate(all_study_ids)
        shard_of_all = np.concatenate(shard_of)
        local_idx_all = np.concatenate(local_idx)

        if study_ids is not None:
            keep = np.isin(study_ids_all, np.asarray(list(study_ids)))
            mask_all, study_ids_all = mask_all[keep], study_ids_all[keep]
            shard_of_all, local_idx_all = shard_of_all[keep], local_idx_all[keep]

        self.mask = mask_all
        self.study_ids = study_ids_all
        self.shard_of = shard_of_all
        self.local_idx = local_idx_all

        aligned = labels_df.reindex(self.study_ids)[FINDINGS]
        if aligned.isna().any().any():
            missing = self.study_ids[aligned.isna().any(axis=1).to_numpy()]
            raise ValueError(f"{len(missing)} cache studies missing labels, e.g. {missing[:5]}")
        self.labels = aligned.to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        shard_idx, row = self.shard_of[i], self.local_idx[i]
        full = self.caches[shard_idx][row]  # (6, 9, 224, 224) uint8
        if self.expand_groups:
            selected, slot_mask_row = expand_slot_groups(full, self.mask[i])
        else:
            g = self.group_index
            selected = full[:, g * SLOT_CACHE_GROUP_SIZE:(g + 1) * SLOT_CACHE_GROUP_SIZE]
            slot_mask_row = self.mask[i]
        images = torch.from_numpy(np.ascontiguousarray(selected)).float() / 255.0
        mask = torch.from_numpy(slot_mask_row)
        label = torch.from_numpy(self.labels[i])
        return images, mask, label


sanity_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, expand_groups=True)
images, mask, label = sanity_ds[0]
print("images:", images.shape, images.dtype, "mask:", mask.shape, "label:", label.shape)
assert images.shape == (N_SLOTS, 3, 224, 224)
assert mask.shape == (N_SLOTS,)
assert not torch.isnan(images).any()
print("SlotCacheDataset(expand_groups=True) sanity check OK")

try:
    SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, expand_groups=True, group_index=0)
    raise AssertionError("expand_groups=True with a non-default group_index should have raised")
except ValueError:
    print("expand_groups=True with group_index=0 correctly raises - OK")